# 📈 Notebook 2: Adaptive Failure Detection (phi accrual)

The previous notebook showed the dilemma: a fixed timeout is either too jumpy or too slow. Hayashibara's **phi accrual failure detector** (used by Cassandra, Akka) takes a different approach:

> Instead of a binary alive/dead, output a continuously-rising **suspicion score** `φ`. Pages or actions trigger when `φ` crosses a threshold.

Idea:

- Keep a sliding window of recent inter-arrival times (heartbeat 1→2, 2→3, ...).
- When asked "is the node alive *right now*?", compute `t_since_last_beat`. The longer it is, the less probable it is under the recent distribution. Convert that probability into a `φ` score: `φ = -log10(P(t_since_last_beat or longer))`.
- A `φ` of 1 ≈ 10% chance the node is just slow. `φ = 8` ≈ one-in-100-million chance — almost certainly dead.

The threshold (typically `φ > 8`) becomes a single tunable knob that *adapts to whatever the network is doing right now*.

## Learning objectives
- Compute φ from a sliding window of inter-arrival times.
- Compare a fixed-timeout detector vs phi accrual on the same trace.

In [ ]:
import math, random, statistics
random.seed(0)

class PhiDetector:
    def __init__(self, window=100):
        self.intervals = []
        self.window = window
        self.last_beat = None

    def heartbeat(self, t):
        if self.last_beat is not None:
            self.intervals.append(t - self.last_beat)
            if len(self.intervals) > self.window:
                self.intervals.pop(0)
        self.last_beat = t

    def phi(self, now):
        if self.last_beat is None or len(self.intervals) < 2:
            return 0.0
        mean = statistics.mean(self.intervals)
        std = max(statistics.pstdev(self.intervals), 1e-3)
        delta = now - self.last_beat
        # Probability of an inter-arrival >= delta under a Normal(mean, std)
        # using the survival function of a Normal: 1 - Phi((delta-mean)/std).
        z = (delta - mean) / std
        # Cheap analytical survival function approximation:
        p = 0.5 * math.erfc(z / math.sqrt(2))
        p = max(p, 1e-12)
        return -math.log10(p)

In [ ]:
# Same trace: heartbeats every ~1s with jitter, node dies at t=30.
TICK, JITTER, DEAD_AT, TOTAL = 1.0, 0.4, 30.0, 40.0
heartbeats = []
t = 0.0
while t < TOTAL:
    t += TICK + random.uniform(-JITTER, JITTER)
    if t >= DEAD_AT: break
    heartbeats.append(t)

# Sample phi every 0.1s
detector = PhiDetector()
phis = []
hb_iter = iter(heartbeats)
next_hb = next(hb_iter, None)
ts, vals = [], []
t = 0.0
while t < TOTAL:
    while next_hb is not None and next_hb <= t:
        detector.heartbeat(next_hb)
        next_hb = next(hb_iter, None)
    ts.append(t); vals.append(detector.phi(t))
    t += 0.1

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
plt.plot(ts, vals, label="phi")
plt.axhline(8, color="red", linestyle="--", label="threshold (phi=8)")
plt.axvline(DEAD_AT, color="grey", linestyle=":", label="actually died")
plt.xlabel("time (s)"); plt.ylabel("phi")
plt.title("Phi accrual: suspicion grows smoothly until it crosses the threshold")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

cross = next((t for t, v in zip(ts, vals) if v > 8), None)
print(f"phi crossed 8 at t={cross:.1f}s (true death at t={DEAD_AT})")

## ✅ Recap

- **Fixed timeout** is simple but you must hand-tune for the worst-case network. Same threshold across a fast LAN and a flaky WAN won't work.
- **Phi accrual** *learns* the normal cadence and gives you one knob (the threshold) that means roughly the same thing everywhere.
- Cassandra defaults to `phi > 8`. Akka uses the same idea for cluster membership.

The mental model: instead of saying "node X is dead", we say "we are increasingly suspicious of node X, and at some confidence we're going to act on that suspicion.